# Audit resolusi spasial Sumatra

Notebook ini sudah berisi seluruh script audit. Tidak perlu menyalin kode dari repository.

**Persiapan:** upload file lokal `data/processed/sumatra_grid_day.parquet` sebagai Kaggle Dataset, lalu attach Dataset itu ke notebook ini. Dataset FIRMS/CHIRPS/ERA5/WorldCover yang sudah diupload boleh tetap disimpan; audit memakai panel olahan, sehingga tidak perlu membaca ulang semua data mentah.

Setelah Dataset panel terpasang, pilih **Run All** pada sesi CPU. Jika DuckDB belum tersedia, aktifkan Internet untuk instalasi dependency. Notebook menghasilkan laporan dan arsip hasil yang bisa diunduh. Tidak ada model Markov atau ambang state yang dilatih/dipilih.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb>=1.4,<2"])
import duckdb
print("DuckDB:", duckdb.__version__)


In [ ]:
from pathlib import Path

# Jika ada beberapa versi panel, isi INPUT_OVERRIDE dengan path yang diinginkan.
INPUT_OVERRIDE = None
candidates = sorted(Path("/kaggle/input").rglob("sumatra_grid_day.parquet"))
if INPUT_OVERRIDE:
    panel_path = Path(INPUT_OVERRIDE)
elif len(candidates) == 1:
    panel_path = candidates[0]
elif not candidates:
    raise FileNotFoundError(
        "Panel belum terpasang. Dataset yang hanya berisi CSV FIRMS dan raster CHIRPS/ERA5/WorldCover "
        "adalah data mentah. Upload sumatra_grid_day.parquet dari folder data/processed di laptop "
        "sebagai Dataset terpisah, attach ke notebook, lalu Run All lagi. "
        "Tidak perlu mengupload ulang Dataset mentah."
    )
else:
    raise ValueError("Ada beberapa panel; isi INPUT_OVERRIDE dengan salah satu path: " + str(candidates))
if not panel_path.is_file():
    raise FileNotFoundError(panel_path)
print("Input:", panel_path)
print("Ukuran:", round(panel_path.stat().st_size / 1_000_000, 1), "MB")


In [ ]:
# Script audit lengkap disimpan terpisah dari Dataset panel.
from pathlib import Path
SCRIPT_PATH = Path("/kaggle/working/spatial_resolution_audit.py")
SCRIPT_PATH.write_text('#!/usr/bin/env python3\n"""Standalone, CPU-only Sumatra spatial audit. Requires DuckDB >= 1.4."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport tempfile\nfrom pathlib import Path\n\nRESOLUTIONS = (0.1, 0.2, 0.25, 0.5)\nENV = (\'temperature_t\', \'precipitation_t\', \'humidity_t\', \'wind_speed_t\', \'rain_t\')\nLEVELS = (10, 20, 30, 50, 100, 200)\n\n\ndef literal(value):\n    return "\'" + str(value).replace("\'", "\'\'") + "\'"\n\n\ndef write_csv(path, rows):\n    with path.open(\'w\', newline=\'\', encoding=\'utf-8\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef record(con, sql):\n    cursor = con.execute(sql)\n    return dict(zip([c[0] for c in cursor.description], cursor.fetchone()))\n\n\ndef export(con, sql, path):\n    con.execute(f\'COPY ({sql}) TO {literal(path)} (HEADER, DELIMITER \\\',\\\')\')\n\n\ndef validate(con):\n    columns = {r[0] for r in con.execute(\'DESCRIBE source\').fetchall()}\n    required = {\'grid_id\', \'date\', \'centroid_lat\', \'centroid_lon\', \'row_index\',\n                \'col_index\', \'hotspot_count\', \'frp_sum\', \'frp_max\'}\n    if required - columns:\n        raise ValueError(f\'Missing required columns: {sorted(required - columns)}\')\n    con.execute(\'CREATE TABLE units AS SELECT DISTINCT grid_id, centroid_lat, centroid_lon, row_index, col_index FROM source\')\n    n = con.execute(\'SELECT count(*) FROM units\').fetchone()[0]\n    if not n or con.execute(\'SELECT count(DISTINCT grid_id) FROM units\').fetchone()[0] != n:\n        raise ValueError(\'Grid coordinates/indices are not persistent or panel is empty\')\n    bad = con.execute(\'\'\'SELECT count(*) FROM units WHERE grid_id IS NULL OR\n        centroid_lat IS NULL OR centroid_lon IS NULL OR row_index IS NULL OR col_index IS NULL OR\n        abs(centroid_lat - (6.0 - row_index * 0.1)) > 1e-7 OR\n        abs(centroid_lon - (95.0 + col_index * 0.1)) > 1e-7 OR\n        row_index NOT BETWEEN 0 AND 120 OR col_index NOT BETWEEN 0 AND 110\'\'\').fetchone()[0]\n    if bad:\n        raise ValueError(\'Input does not match the verified ERA5-centered 0.1-degree lattice\')\n    con.execute(\'CREATE TABLE dates AS SELECT DISTINCT CAST(date AS DATE) AS date FROM source\')\n    d = record(con, \'SELECT count(*) AS n, min(date) AS first, max(date) AS last FROM dates\')\n    if d[\'first\'] is None or (d[\'last\'] - d[\'first\']).days + 1 != d[\'n\']:\n        raise ValueError(\'Panel dates must be non-null and consecutive\')\n    months = con.execute("SELECT DISTINCT date_trunc(\'month\', date)::DATE FROM dates ORDER BY 1").fetchall()\n    for (month,) in months:\n        where = f"date >= DATE {literal(month)} AND date < DATE {literal(month)} + INTERVAL \'1 month\'"\n        stats = record(con, f\'\'\'SELECT count(*) AS n, count(DISTINCT CAST(date AS DATE)) AS days,\n            count(*) FILTER (WHERE grid_id IS NULL OR date IS NULL OR hotspot_count IS NULL OR\n                hotspot_count < 0 OR hotspot_count != floor(hotspot_count) OR frp_sum IS NULL OR\n                NOT isfinite(frp_sum) OR frp_sum < 0 OR\n                (hotspot_count = 0 AND (frp_sum != 0 OR frp_max IS NOT NULL)) OR\n                (hotspot_count > 0 AND (frp_max IS NULL OR NOT isfinite(frp_max) OR frp_max < 0))) AS bad\n            FROM source WHERE {where}\'\'\')\n        dup = con.execute(f\'SELECT count(*) FROM (SELECT grid_id, date FROM source WHERE {where} GROUP BY 1,2 HAVING count(*) != 1)\').fetchone()[0]\n        if stats[\'bad\'] or dup or stats[\'n\'] != n * stats[\'days\']:\n            raise ValueError(f\'Invalid, duplicated, or incomplete source panel in {month}\')\n    if con.execute(\'SELECT count(*) FROM source WHERE date IS NULL\').fetchone()[0]:\n        raise ValueError(\'Null date\')\n    return columns, n, d, months\n\n\ndef audit_resolution(args, resolution, work):\n    import duckdb\n    con = duckdb.connect(str(work / \'audit.duckdb\'))\n    try:\n        con.execute(f\'SET memory_limit = {literal(args.memory_limit)}\')\n        con.execute(f\'SET temp_directory = {literal(work / "spill")}\')\n        con.execute(f\'SET threads = {args.threads}\')\n        con.execute(\'SET preserve_insertion_order = false\')\n        con.execute(f\'CREATE VIEW source AS SELECT * FROM read_parquet({literal(args.input)})\')\n        columns, n_source, dates, months = validate(con)\n        env = [c for c in ENV if c in columns]\n        # Integer arithmetic avoids floating point ties at target boundaries.\n        milli = round(resolution * 1000)\n        con.execute(f\'\'\'CREATE TABLE mapping AS SELECT *,\n            CAST(floor((50 + col_index * 100)::DOUBLE / {milli}) AS INTEGER) AS coarse_col,\n            CAST(floor((50 + row_index * 100)::DOUBLE / {milli}) AS INTEGER) AS coarse_row\n            FROM units\'\'\')\n        con.execute("ALTER TABLE mapping ADD COLUMN coarse_grid_id VARCHAR")\n        con.execute("UPDATE mapping SET coarse_grid_id = \'SUM_R\' || coarse_row || \'_C\' || coarse_col")\n        if con.execute(\'SELECT count(DISTINCT coarse_grid_id) FROM mapping\').fetchone()[0] != n_source and resolution == 0.1:\n            raise ValueError(\'Reference resolution failed to preserve base cells\')\n        label = f\'{resolution:g}\'\n        export(con, \'SELECT * FROM mapping ORDER BY grid_id\', args.output / f\'grid_mapping_{label}.csv\')\n        classes = []\n        if \'landcover_code\' in columns:\n            # Static category must be invariant per source grid; numeric codes are never averaged.\n            bad = con.execute(\'\'\'SELECT count(*) FROM (SELECT grid_id FROM source GROUP BY grid_id\n                HAVING count(DISTINCT landcover_code) > 1 OR\n                (count(landcover_code) > 0 AND count(landcover_code) != count(*)))\'\'\').fetchone()[0]\n            if bad:\n                raise ValueError(\'Land-cover categories change or have inconsistent missingness across dates\')\n            classes = [r[0] for r in con.execute(\'SELECT DISTINCT landcover_code FROM source WHERE landcover_code IS NOT NULL ORDER BY 1\').fetchall()]\n            if any(not math.isfinite(float(c)) or float(c) != int(c) or int(c) < 0 for c in classes):\n                raise ValueError(\'Land-cover category must be a nonnegative integer code\')\n        aggregate = [\n            \'m.coarse_grid_id AS grid_id\', \'CAST(s.date AS DATE) AS date\',\n            \'count(*) AS n_source_cells\', \'sum(s.hotspot_count)::BIGINT AS hotspot_count\',\n            \'sum(s.frp_sum) AS frp_sum\', \'max(s.frp_max) AS frp_max\',\n        ]\n        for c in env:\n            aggregate += [f\'avg(s.{c}) AS {c}\', f\'count(s.{c}) AS {c}_n_observed\']\n        if \'landcover_code\' in columns:\n            aggregate += [\'count(s.landcover_code) AS landcover_n_observed\']\n            aggregate += [f\'avg(CASE WHEN s.landcover_code IS NULL THEN NULL ELSE (s.landcover_code = {int(c)})::INTEGER END) AS landcover_fraction_{int(c)}\' for c in classes]\n        select = \', \'.join(aggregate)\n        for index, (month,) in enumerate(months):\n            sql = f\'\'\'SELECT {select} FROM source s JOIN mapping m USING (grid_id)\n                WHERE s.date >= DATE {literal(month)} AND s.date < DATE {literal(month)} + INTERVAL \'1 month\'\n                GROUP BY m.coarse_grid_id, CAST(s.date AS DATE)\'\'\'\n            con.execute((\'CREATE TABLE panel AS \' if index == 0 else \'INSERT INTO panel \') + sql)\n        con.execute(\'\'\'CREATE TABLE sparsity AS SELECT grid_id, count(*) AS total_days,\n            count(*) FILTER (WHERE hotspot_count > 0) AS active_days,\n            count(*) FILTER (WHERE hotspot_count = 0) AS inactive_days,\n            avg((hotspot_count > 0)::INTEGER) AS active_fraction,\n            sum(hotspot_count) AS total_hotspots, sum(frp_sum) AS total_frp\n            FROM panel GROUP BY grid_id\'\'\')\n        stats = record(con, \'\'\'SELECT count(*) AS n_grid_days,\n            count(DISTINCT date) AS n_dates,\n            count(*) FILTER (WHERE hotspot_count > 0) AS active_grid_days,\n            sum(hotspot_count) AS total_hotspots, sum(frp_sum) AS total_frp FROM panel\'\'\')\n        row = {\'resolution\': resolution, **stats}\n        row.update(record(con, \'\'\'SELECT count(*) AS n_units,\n            count(*) FILTER (WHERE active_days = 0) AS zero_event_units FROM sparsity\'\'\'))\n        row[\'active_grid_day_pct\'] = 100 * row[\'active_grid_days\'] / row[\'n_grid_days\']\n        if row[\'n_grid_days\'] != row[\'n_units\'] * dates[\'n\']:\n            raise ValueError(\'Coarse panel completeness failed\')\n        original = record(con, \'SELECT sum(hotspot_count) AS total_hotspots, sum(frp_sum) AS total_frp FROM source\')\n        if row[\'total_hotspots\'] != original[\'total_hotspots\'] or not math.isclose(row[\'total_frp\'], original[\'total_frp\'], rel_tol=1e-10, abs_tol=1e-6):\n            raise ValueError(\'Coarsening failed event/FRP conservation\')\n        for column, quantiles in [(\'active_days\', (0, .1, .25, .5, .75, .9, 1)), (\'total_hotspots\', (0, .25, .5, .75, .9, 1))]:\n            for q in quantiles:\n                prefix = \'min\' if q == 0 else \'max\' if q == 1 else \'median\' if q == .5 else f\'p{round(q*100)}\'\n                row[f\'{prefix}_{column}\'] = con.execute(f\'SELECT quantile_cont({column}, {q}) FROM sparsity\').fetchone()[0]\n        for level in LEVELS:\n            count = con.execute(f\'SELECT count(*) FROM sparsity WHERE active_days >= {level}\').fetchone()[0]\n            row[f\'units_ge_{level}_active_days\'] = count\n            row[f\'pct_units_ge_{level}_active_days\'] = 100 * count / row[\'n_units\']\n        intensity = []\n        for column in (\'hotspot_count\', \'frp_sum\', \'frp_max\'):\n            item = {\'resolution\': resolution, \'variable\': column}\n            item.update(record(con, f\'SELECT count({column}) AS n, \' + \', \'.join(f\'quantile_cont({column}, {q}) AS p{round(100*q)}\' for q in (.25, .5, .75, .9, .95, .99)) + f\', max({column}) AS max FROM panel WHERE hotspot_count > 0\'))\n            intensity.append(item)\n        # Record coverage so averaging available cells cannot conceal missing inputs.\n        coverage = {\'resolution\': resolution, \'environmental_columns\': env, \'absent_environmental_columns\': [c for c in ENV if c not in env], \'landcover_category_codes\': classes, \'landcover_column_available\': \'landcover_code\' in columns}\n        for c in env:\n            coverage[c] = record(con, f\'SELECT sum(n_source_cells - {c}_n_observed) AS missing_source_grid_days, count(*) FILTER (WHERE {c} IS NULL) AS all_missing_coarse_grid_days FROM panel\')\n        if \'landcover_code\' in columns:\n            coverage[\'landcover\'] = record(con, \'SELECT sum(n_source_cells - landcover_n_observed) AS missing_source_grid_days, count(*) FILTER (WHERE landcover_n_observed = 0) AS all_missing_coarse_grid_days FROM panel\')\n        export(con, \'SELECT * FROM sparsity ORDER BY grid_id\', args.output / f\'per_unit_sparsity_{label}.csv\')\n        if args.keep_panels:\n            con.execute(f\'COPY panel TO {literal(args.output / ("panel_" + label + ".parquet"))} (FORMAT PARQUET, COMPRESSION ZSTD)\')\n        return row, intensity, coverage, {\'start_date\': str(dates[\'first\']), \'end_date\': str(dates[\'last\']), \'n_source_units\': n_source, \'n_source_dates\': dates[\'n\']}\n    finally:\n        con.close()\n\n\ndef report(rows, intensities, metadata, args):\n    passing = [r for r in rows if r[\'median_active_days\'] >= args.min_median_active_days and r[\'pct_units_ge_20_active_days\'] >= args.min_pct_units_ge_20]\n    chosen = min(passing, key=lambda r: r[\'resolution\']) if passing else None\n    lines = [\'# Spatial-resolution feasibility audit\', \'\', \'## Methodology\', \'\',\n        f"Input: `{args.input.name}`; {metadata[\'n_source_units\']:,} source units, {metadata[\'n_source_dates\']:,} consecutive dates ({metadata[\'start_date\']} to {metadata[\'end_date\']}). Input uniqueness, completeness, lattice coordinates, static land-cover invariance and event/FRP conservation are checked.", \'\',\n        \'Target bins share the verified northwest edge (longitude 94.95°, latitude 6.05°). Whole 0.1° cells are assigned by centroid using integer row/column arithmetic. The 0.25° bins do not nest into 0.1° cells: allocation alternates numbers of source cells and is an approximation, not exact geometric regridding. Only existing Sumatra source cells contribute. Each coarse unit is the union of its assigned source cells; full target rectangles are not added to the domain. No temporal aggregation occurs.\', \'\',\n        \'## Aggregation rules\', \'\',\n        \'Hotspot count and FRP sum are summed; FRP max is the maximum (null for inactive days). Available continuous environmental values are averaged equally over source cells; all-missing groups stay null. Observed-member counts retain missingness. Source cells have equal angular size, so these are equal-cell rather than exact land-area-weighted means. WorldCover codes become category shares among known source cells, not numeric-code averages or true within-cell land-cover fractions. No missing category is imputed. Existing fraction columns are not present in the verified input.\', \'\',\n        \'Each resolution is processed separately in a disk-backed DuckDB database, aggregating one calendar month at a time. Temporary panels are removed after diagnostics unless `--keep-panels` is requested. Quantiles use exact continuous interpolation over active coarse grid-days; no state thresholds are selected.\', \'\',\n        \'## Observed comparison\', \'\',\n        \'| Resolution | Units | Active grid-days (%) | Zero-event units | Median active days | Units ≥20 (%) | Units ≥50 (%) | Units ≥100 (%) |\',\n        \'|---|---:|---:|---:|---:|---:|---:|---:|\']\n    for r in rows:\n        lines.append(f"| {r[\'resolution\']:g}° | {r[\'n_units\']:,} | {r[\'active_grid_day_pct\']:.3f} | {r[\'zero_event_units\']:,} | {r[\'median_active_days\']:.1f} | {r[\'pct_units_ge_20_active_days\']:.1f} | {r[\'pct_units_ge_50_active_days\']:.1f} | {r[\'pct_units_ge_100_active_days\']:.1f} |")\n    lines += [\'\', \'All requested counts and percentiles, including ≥10/30/200 active days, are in the comparison CSV; per-unit diagnostics and source-to-coarse mappings are exported separately.\', \'\', \'## Active-day intensity distributions\', \'\', \'| Resolution | Variable | n | p25 | p50 | p75 | p90 | p95 | p99 | max |\', \'|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|\']\n    for item in intensities:\n        values = [\'NA\' if item[k] is None else f\'{item[k]:.6g}\' for k in (\'n\', \'p25\', \'p50\', \'p75\', \'p90\', \'p95\', \'p99\', \'max\')]\n        lines.append(f"| {item[\'resolution\']:g}° | {item[\'variable\']} | " + \' | \'.join(values) + \' |\')\n    lines += [\'\', \'## Observed sparsity\', \'\']\n    for r in rows:\n        lines.append(f"- {r[\'resolution\']:g}°: {r[\'zero_event_units\']:,}/{r[\'n_units\']:,} units have no events; the central 50% have {r[\'p25_active_days\']:.1f}–{r[\'p75_active_days\']:.1f} active days.")\n    lines += [\'\', \'## Recommendation\', \'\', f\'Exploratory screening rule: median active days ≥{args.min_median_active_days:g}, and ≥{args.min_pct_units_ge_20:g}% of units have at least 20 active days. These are configurable audit heuristics, not scientific eligibility thresholds or evidence that four-state models are estimable.\']\n    lines.append(f"The finest evaluated resolution passing this screen is **{chosen[\'resolution\']:g}°**, provisionally recommended for state-definition assessment." if chosen else \'No evaluated resolution passes the screen. No resolution is recommended automatically; retain the research design for review with the observed sparsity evidence.\')\n    lines += [\'\', \'## Remaining uncertainty\', \'\', \'Active-day density does not establish sufficient observations per current state or diversity of next states. Low/Moderate/High definitions, training-period counts, transition frequencies, covariate availability, and usable complete-case samples still need assessment. Coarsening can dilute local activity and hide missing covariates. The 0.25° centroid allocation and equal-cell weighting warrant sensitivity checks. No predictive model has been fitted.\', \'\', \'Covariate missingness and aggregation coverage are recorded in `spatial_resolution_run_metadata.json`.\', \'\']\n    (args.output / \'spatial_resolution_feasibility.md\').write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    return None if chosen is None else chosen[\'resolution\']\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\', required=True, type=Path)\n    parser.add_argument(\'--output\', required=True, type=Path)\n    parser.add_argument(\'--resolutions\', nargs=\'+\', type=float, default=list(RESOLUTIONS))\n    parser.add_argument(\'--overwrite\', action=\'store_true\')\n    parser.add_argument(\'--keep-panels\', action=\'store_true\')\n    parser.add_argument(\'--memory-limit\', default=\'4GB\')\n    parser.add_argument(\'--threads\', type=int, default=2)\n    parser.add_argument(\'--min-median-active-days\', type=float, default=50)\n    parser.add_argument(\'--min-pct-units-ge-20\', type=float, default=50)\n    args = parser.parse_args(argv)\n    args.input = args.input.resolve(); args.output = args.output.resolve()\n    if not args.input.is_file():\n        parser.error(\'Input Parquet file does not exist\')\n    if any(r not in RESOLUTIONS for r in args.resolutions) or len(set(args.resolutions)) != len(args.resolutions):\n        parser.error(\'Use unique resolutions from 0.1 0.2 0.25 0.5\')\n    if args.threads < 1 or args.min_median_active_days < 0 or not 0 <= args.min_pct_units_ge_20 <= 100:\n        parser.error(\'Invalid thread count or screening criteria\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    if any(args.output.iterdir()) and not args.overwrite:\n        parser.error(\'Output directory is not empty; choose a new directory or use --overwrite\')\n    if args.input.parent == args.output or args.output in args.input.parents:\n        parser.error(\'Output directory must not contain the source panel\')\n    try:\n        import duckdb\n    except ImportError:\n        parser.error(\'Install DuckDB: python -m pip install "duckdb>=1.4,<2"\')\n    rows, intensities, coverage = [], [], []\n    metadata = {}\n    for resolution in sorted(args.resolutions):\n        print(f\'Auditing {resolution:g} degrees...\', flush=True)\n        with tempfile.TemporaryDirectory(prefix=\'spatial_audit_\', dir=args.output) as scratch:\n            row, detail, cov, metadata = audit_resolution(args, resolution, Path(scratch))\n        rows.append(row); intensities.extend(detail); coverage.append(cov)\n    write_csv(args.output / \'spatial_resolution_feasibility.csv\', rows)\n    write_csv(args.output / \'active_day_intensity.csv\', intensities)\n    recommendation = report(rows, intensities, metadata, args)\n    metadata.update({\'input\': str(args.input), \'input_bytes\': args.input.stat().st_size,\n                     \'duckdb_version\': duckdb.__version__, \'resolutions\': sorted(args.resolutions),\n                     \'memory_limit\': args.memory_limit, \'threads\': args.threads,\n                     \'coverage\': coverage, \'recommended_resolution\': recommendation,\n                     \'recommendation_screen\': {\'min_median_active_days\': args.min_median_active_days, \'min_pct_units_ge_20\': args.min_pct_units_ge_20}})\n    (args.output / \'spatial_resolution_run_metadata.json\').write_text(json.dumps(metadata, indent=2, default=str) + \'\\n\')\n    print(f\'Reports written to {args.output}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")
print("Script siap:", SCRIPT_PATH)


In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

# Folder baru tiap run mencegah tercampurnya hasil versi script yang berbeda.
results_dir = Path("/kaggle/working") / ("results_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
subprocess.run([
    sys.executable, str(SCRIPT_PATH),
    "--input", str(panel_path),
    "--output", str(results_dir),
    "--resolutions", "0.1", "0.2", "0.25", "0.5",
    "--memory-limit", "4GB", "--threads", "2",
], check=True)
print("Hasil:", results_dir)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, FileLink
import tarfile

comparison = pd.read_csv(results_dir / "spatial_resolution_feasibility.csv")
display(comparison)
display(Markdown((results_dir / "spatial_resolution_feasibility.md").read_text()))
archive = Path("/kaggle/working") / (results_dir.name + ".tar.gz")
with tarfile.open(archive, "w:gz") as output:
    for file in sorted(results_dir.iterdir()):
        if file.is_file():
            output.add(file, arcname=file.name)
print("Unduh arsip ini; file juga ada di panel Output Kaggle:", archive)
display(FileLink(str(archive)))
